# LivePortrait Server (chạy nền trên Colab)

Notebook này KHÔNG dùng để bấm tay từng video. Nó dựng LivePortrait + một **API server** rồi mở **cloudflared tunnel** để máy của bạn gọi vào.

Quy trình tự động (do orchestrator/bot lái):
1. Đổi runtime sang **T4 GPU** (Runtime → Change runtime type → T4 GPU).
2. **Run all** — các cell tự setup rồi chạy server.
3. Cell cuối in ra dòng `PUBLIC_URL=https://....trycloudflare.com` — bot đọc dòng này.
4. Server nhận POST `/generate` với `{source_url, driving_url}` → trả về file mp4.

## 1) Kiểm tra GPU (phải thấy Tesla T4 hoặc tương đương)

In [ ]:
!nvidia-smi

Mon Aug 24 08:29:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Clone LivePortrait + cài thư viện + tải weights (gộp setup)

In [ ]:
import os, subprocess, re

# ===== Clone LivePortrait =====
REPOS = [
    'https://github.com/KwaiVGI/LivePortrait',
    'https://github.com/KlingTeam/LivePortrait',
    'https://github.com/KlingAIResearch/LivePortrait',
]
if not os.path.isdir('/content/LivePortrait'):
    ok = False
    for url in REPOS:
        print('Thu clone:', url)
        r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/LivePortrait'])
        if r.returncode == 0:
            ok = True; print('OK ->', url); break
    assert ok, 'Khong clone duoc repo nao.'
os.chdir('/content/LivePortrait')
print('CWD =', os.getcwd())

# ===== Cai thu vien =====
# Gop requirements.txt + requirements_base.txt (file dau chi co dong '-r' tro sang
# file sau -> neu chi doc requirements.txt se KHONG cai gi ca => loi 'No module named tyro').
# Bo torch/torchvision/torchaudio (giu ban co san cua Colab) va gradio (chi dung cho
# demo web app.py, khong can cho inference.py -> keo theo nhieu dep nang lam cham).
# Doi == thanh >= de tranh loi 'no matching distribution' khi ban ghim cung khong con wheel.
lines = []
for fname in ('requirements.txt', 'requirements_base.txt'):
    if os.path.exists(fname):
        with open(fname) as f:
            lines += f.readlines()
SKIP = r'\s*(torch|torchvision|torchaudio|gradio|-r\s)'
kept = [l for l in lines if l.strip() and not re.match(SKIP, l)]
kept = [re.sub(r'==', '>=', l) for l in kept]
with open('requirements_colab.txt', 'w') as f:
    f.writelines(kept)
print(''.join(kept))
!pip install -q -r requirements_colab.txt
!pip install -q -U huggingface_hub
import importlib
for m in ('tyro', 'onnxruntime'):
    print(m, '->', 'OK' if importlib.util.find_spec(m) else 'THIEU')
print('Cai xong dependencies LivePortrait.')

Thu clone: https://github.com/KwaiVGI/LivePortrait
OK -> https://github.com/KwaiVGI/LivePortrait
CWD = /content/LivePortrait
onnxruntime-gpu>=1.18.0
transformers>=4.38.0
numpy>=1.26.4
pyyaml>=6.0.1
opencv-python>=4.10.0.84
scipy>=1.13.1
imageio>=2.34.2
lmdb>=1.4.1
tqdm>=4.66.4
rich>=13.7.1
ffmpeg-python>=0.2.0
onnx>=1.16.1
scikit-image>=0.24.0
albumentations>=1.4.10
matplotlib>=3.9.0
imageio-ffmpeg>=0.5.1
tyro>=0.8.5
pykalman>=0.9.7
pillow>=10.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.2/202.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 48.9 M

In [ ]:
from huggingface_hub import snapshot_download

HF_REPOS = ['KwaiVGI/LivePortrait', 'KlingTeam/LivePortrait']
done = False
for rid in HF_REPOS:
    try:
        print('Tai weights tu HuggingFace:', rid)
        snapshot_download(repo_id=rid, local_dir='pretrained_weights',
                          allow_patterns=['*.pth', '*.onnx', '*.yaml', '*.json', '*.bin', '*.txt', '**/*'],
                          ignore_patterns=['*.git*', 'README.md', 'docs/*'])
        done = True; print('OK ->', rid); break
    except Exception as e:
        print('Loi voi', rid, ':', e)
assert done, 'Khong tai duoc weights.'
print('Weights san sang.')

Tai weights tu HuggingFace: KwaiVGI/LivePortrait


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

OK -> KwaiVGI/LivePortrait
Weights san sang.


## 3) Cài server (FastAPI) + cloudflared

In [ ]:
!pip install -q fastapi 'uvicorn[standard]' requests python-multipart
# Tai binary cloudflared (khong can tai khoan)
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
print('cloudflared:', subprocess.run(['/usr/local/bin/cloudflared', '--version'], capture_output=True, text=True).stdout.strip())

cloudflared: cloudflared version 2026.8.2 (built 2026-08-14-12:17 UTC)


## 4) Định nghĩa hàm inference LivePortrait

In [ ]:
import glob, os, re, subprocess, shutil, uuid

LP_DIR = '/content/LivePortrait'

# Tìm sẵn 1 lần tên flag tắt pasteback (khác nhau giữa các bản repo)
_h = subprocess.run(['python', 'inference.py', '--help'], cwd=LP_DIR,
                    capture_output=True, text=True)
_flags = sorted(set(re.findall(r'--[A-Za-z0-9_\-]*pasteback[A-Za-z0-9_\-]*', _h.stdout + _h.stderr)))
PASTE_OFF = next((c for c in _flags if c.lower().startswith('--no')), (_flags or [None])[0])
print('Flag tắt pasteback:', PASTE_OFF)

def run_liveportrait(src_path, drv_path, scale=2.0, paste_back=False):
    """Chạy LivePortrait, trả về đường dẫn mp4 kết quả (đã mux audio nếu có)."""
    cmd = ['python', 'inference.py', '-s', src_path, '-d', drv_path,
           '--flag_crop_driving_video', '--scale', str(scale)]
    if not paste_back and PASTE_OFF:
        cmd.append(PASTE_OFF)
    print('Lệnh:', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=LP_DIR, capture_output=True, text=True)

    anim = os.path.join(LP_DIR, 'animations')
    mp4s = [f for f in glob.glob(os.path.join(anim, '*.mp4')) if '_concat' not in f]
    if not mp4s:
        raise RuntimeError('Inference lỗi:\n' + p.stderr[-2000:])
    pool = [f for f in mp4s if '_with_audio' in f] or mp4s
    result = max(pool, key=os.path.getmtime)

    os.makedirs('/content/results', exist_ok=True)
    final = f'/content/results/{uuid.uuid4().hex[:8]}.mp4'
    if '_with_audio' in result:
        shutil.copy(result, final)
    else:
        has_audio = os.system(
            f'ffprobe -v error -select_streams a -show_entries stream=codec_type '
            f'-of csv=p=0 "{drv_path}" | grep -q audio') == 0
        if has_audio:
            os.system(f'ffmpeg -y -i "{result}" -i "{drv_path}" -map 0:v:0 -map 1:a:0 '
                      f'-c:v copy -c:a aac -shortest "{final}"')
        else:
            shutil.copy(result, final)
    return final

print('Hàm run_liveportrait sẵn sàng.')

Flag tắt pasteback: --no-flag-pasteback
Hàm run_liveportrait sẵn sàng.


## 5) Chạy server + mở tunnel
Cell này chạy MÃI (giữ kernel bận = giữ phiên). Đợi dòng `PUBLIC_URL=...` xuất hiện.

In [ ]:
import os, re, time, threading, subprocess, uuid, traceback
from typing import Optional
from urllib.parse import urlparse, urlsplit
import requests
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn

PORT = 8000
app = FastAPI(title='LivePortrait Server')

class Job(BaseModel):
    source_url: str
    driving_url: str
    scale: float = 2.0
    paste_back: bool = False
    job_id: Optional[str] = None   # caller (aiService) tự đặt id để poll bằng id đó

def _download(url, dst):
    # Giả lập trình duyệt để qua mặt chặn hotlink/403 ở nhiều trang
    sp = urlsplit(url)
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                       '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'),
        'Accept': 'image/avif,image/webp,image/*,video/*,*/*;q=0.8',
        'Referer': f'{sp.scheme}://{sp.netloc}/',
    }
    r = requests.get(url, stream=True, timeout=180, headers=headers)
    r.raise_for_status()
    with open(dst, 'wb') as f:
        for chunk in r.iter_content(1 << 16):
            f.write(chunk)

def _ext(url, default):
    e = os.path.splitext(urlparse(url).path)[1].lower()
    return e if e else default

# ===== JOB STORE (async) =================================================
# Tránh lỗi Cloudflare 524: /generate KHÔNG render đồng bộ nữa. Thay vào đó nó
# tạo job_id, trả NGAY, rồi render ở thread nền. Client (aiService) poll
# /jobs/{id} tới khi 'done' rồi tải kết quả ở /jobs/{id}/result. Mỗi request
# đều trả nhanh (<100s) nên không bao giờ chạm trần timeout của Cloudflare.
JOBS = {}
JOBS_LOCK = threading.Lock()
RUN_LOCK = threading.Lock()   # serialize GPU: chỉ render 1 job 1 lúc
JOB_TTL = 3600                 # giữ kết quả 1h rồi dọn file + entry

def _prune_jobs_locked():
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('result') and os.path.exists(v['result']):
                os.remove(v['result'])
        except Exception:
            pass

def _new_job(jid=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_jobs_locked()
        JOBS[jid] = {'status': 'processing', 'result': None, 'media_type': None,
                     'filename': None, 'error': None, 'created': time.time()}
    return jid

def _set_done(jid, path, media_type, filename):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid].update(status='done', result=path,
                             media_type=media_type, filename=filename)

def _set_error(jid, msg):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid].update(status='error', error=str(msg)[:500])

def _run_async(jid, work):
    # work() -> (out_path, media_type, filename)
    try:
        with RUN_LOCK:
            out, media_type, filename = work()
        _set_done(jid, out, media_type, filename)
        print(f'[server] xong job {jid[:8]} -> {out}')
    except Exception as e:
        traceback.print_exc()
        _set_error(jid, e)
        print(f'[server] LOI job {jid[:8]}: {e}')

@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        return {'job_id': job_id, 'status': j['status'], 'error': j['error']}

@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        status, path, mt, fn, err = j['status'], j['result'], j['media_type'], j['filename'], j['error']
    if status == 'processing':
        raise HTTPException(status_code=409, detail='job chua xong')
    if status == 'error':
        raise HTTPException(status_code=500, detail=err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=410, detail='ket qua khong con (da bi don)')
    return FileResponse(path, media_type=mt, filename=fn)
# =========================================================================

@app.get('/health')
def health():
    return {'ok': True}

@app.post('/generate')
def generate(job: Job):
    jid = _new_job(job.job_id)
    print(f'[server] nhan request /generate ({jid[:8]}) -> chay nen, tra job_id ngay.')
    def work():
        uid = jid[:8]
        os.makedirs('/content/jobs', exist_ok=True)
        src = f'/content/jobs/{uid}_src{_ext(job.source_url, ".png")}'
        drv = f'/content/jobs/{uid}_drv{_ext(job.driving_url, ".mp4")}'
        _download(job.source_url, src)
        _download(job.driving_url, drv)
        out = run_liveportrait(src, drv, job.scale, job.paste_back)
        return out, 'video/mp4', f'{uid}.mp4'
    threading.Thread(target=_run_async, args=(jid, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing'}

# 1) Khởi động uvicorn trong thread nền
def _serve():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning')
threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

# 2) Mở cloudflared tunnel, đọc URL công khai từ log
proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}',
     '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        # Dòng MỐC để orchestrator/bot đọc:
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

# 3) Giữ cell sống (giữ phiên Colab + tunnel)
print('Server đang chạy. Giữ cell này mở. URL:', public_url)
print('>>> SAN SANG - POST /generate (tra job_id), GET /jobs/{id}, GET /jobs/{id}/result.')
print('>>> CHUA render video nao (KHONG ton GPU cho toi khi co request that).')
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')


2026-08-24T08:31:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-24T08:31:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-24T08:31:44Z INF +--------------------------------------------------------------------------------------------+
2026-08-24T08:31:44Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-24T08:31:44Z INF |  https://research-indiana-sections-cult.trycloudflare.